In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import random

/global/homes/b/boyanyin/.conda/envs/sompz/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
input_file = 'e2e_catalog_noshear.parquet'

parquet_file = pq.ParquetFile(input_file)

df_head = parquet_file.read_row_group(0).to_pandas().head(100)

#print(df_head)

In [4]:
input_file = 'e2e_catalog_noshear.parquet'
output_file = 'e2e_catalog_noshear_random1M.parquet'
sample_size = 1000000

pf = pq.ParquetFile(input_file)
total_rows = pf.metadata.num_rows

target_indices = np.array(sorted(random.sample(range(total_rows), sample_size)))
current_offset = 0

with pq.ParquetWriter(output_file, pf.schema_arrow) as writer:
    for i in range(pf.num_row_groups):
        chunk_size = pf.metadata.row_group(i).num_rows
        chunk_start = current_offset
        chunk_end = current_offset + chunk_size
        
        in_this_chunk = (target_indices >= chunk_start) & (target_indices < chunk_end)
        indices_in_chunk = target_indices[in_this_chunk]
        
        local_indices = indices_in_chunk - chunk_start
            
        table = pf.read_row_group(i)
        sampled_table = table.take(pa.array(local_indices))
        writer.write_table(sampled_table)
        
        current_offset += chunk_size

In [ ]:
with pq.ParquetWriter(output_file, pf.schema_arrow) as writer:
    print(pf.num_row_groups)
    print(pf.metadata.row_group(0).num_rows)

In [2]:
print(df_head.keys())

Index(['flux_pgauss_LSST_u', 'flux_err_pgauss_LSST_u', 'flux_pgauss_LSST_g',
       'flux_err_pgauss_LSST_g', 'flux_pgauss_LSST_r',
       'flux_err_pgauss_LSST_r', 'flux_pgauss_LSST_i',
       'flux_err_pgauss_LSST_i', 'flux_pgauss_LSST_z',
       'flux_err_pgauss_LSST_z', 'flux_pgauss_LSST_y',
       'flux_err_pgauss_LSST_y', 'flux_pgauss_Y', 'flux_err_pgauss_Y',
       'flux_pgauss_J', 'flux_err_pgauss_J', 'flux_pgauss_H',
       'flux_err_pgauss_H', 'flux_gold_LSST_u', 'flux_err_gold_LSST_u',
       'flux_gold_LSST_g', 'flux_err_gold_LSST_g', 'flux_gold_LSST_r',
       'flux_err_gold_LSST_r', 'flux_gold_LSST_i', 'flux_err_gold_LSST_i',
       'flux_gold_LSST_z', 'flux_err_gold_LSST_z', 'flux_gold_LSST_y',
       'flux_err_gold_LSST_y', 'flux_gold_Y', 'flux_err_gold_Y', 'flux_gold_J',
       'flux_err_gold_J', 'flux_gold_H', 'flux_err_gold_H', 'g1', 'g2',
       'objectid', 'ra', 'dec', 'z', 'snr', 'pgauss_s2n', 'reff',
       'pgauss_T_ratio'],
      dtype='object')


In [3]:
num_rows = parquet_file.metadata.num_rows
print(f"Total number of rows: {num_rows}")

Total number of rows: 52220362
